# 🤖 Agent Development - Risk Analyst & Compliance Officer

Welcome to Phase 2 & 3 of the project! In this notebook, you'll develop and test both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 🎯 Learning Objectives
- Implement Chain-of-Thought prompting for systematic reasoning
- Build ReACT prompting for structured action-taking
- Handle structured JSON outputs from LLMs
- Create robust error handling and validation
- Test agents with real financial data

## 🚀 Setup and Environment

In [1]:
!find . -name "__pycache__" -type d -exec rm -rf {} +

In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [3]:
%reload_ext autoreload

In [4]:
# Import required libraries
import os
import sys
import json
import openai
from dotenv import load_dotenv
from datetime import datetime

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path:", os.path.abspath('../src'))

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Source directory added to Python path: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/src


In [5]:
import sys
import os

# Get the absolute path to the 'starter' directory
# (Assuming your notebook is in starter/notebooks)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add 'starter' to sys.path so 'src' is importable
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now test the import to ensure it works in the notebook
try:
    from src.foundation_sar import RiskAnalystOutput
    print("✅ Success: Module loaded correctly.")
except ImportError as e:
    print(f"❌ Still failing: {e}")

✅ Success: Module loaded correctly.


In [6]:
# OpenAI Setup for Vocareum
import openai

# Option 1: Use the helper function from src package (recommended)
# Uncomment this after implementing foundation_sar.py:
# import create_vocareum_openai_client
# client = create_vocareum_openai_client()

# Option 2: Manual setup (for early development)
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")
    # print("\n💡 Tip: After implementing foundation_sar.py, you can use:")
    # print("   from src import create_vocareum_openai_client")
#     print("   client = create_vocareum_openai_client()")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: voc-1750...0392
📍 Base URL: https://openai.vocareum.com/v1


## 📊 Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [7]:
# TODO: Import your implemented foundation components
# Uncomment and modify these imports once you've implemented foundation_sar.py
print("📋 TODO: Import foundation components after implementing foundation_sar.py")

# Import your implemented foundation components
from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)

print("✅ Foundation components successfully imported from src/foundation_sar.py!")
print("✅ Once imported, you can create sample cases for agent testing")

📋 TODO: Import foundation components after implementing foundation_sar.py
✅ Foundation components successfully imported from src/foundation_sar.py!
✅ Once imported, you can create sample cases for agent testing


In [8]:
# TODO: Test foundation components
# Once you've implemented foundation_sar.py, use this cell to:

# 1. Setup deterministic paths based on the notebook location
# Calculate the path to the audit_logs directory
notebook_dir = os.getcwd()

print("📋 TODO: Test foundation components")
print("1. Load CSV data from ../data/")
customers_df, accounts_df, transactions_df = load_csv_data("../data")
print("2. Create ExplainabilityLogger instance")
logger = ExplainabilityLogger("../outputs/audit_logs/agent_development.jsonl")
loader = DataLoader(logger)
print("3. Create DataLoader instance")
print("4. Generate sample case for agent testing")

# 3. Pull a sample customer (using our active finder to ensure they have history)
# 4. Find an active customer ID who actually has transactions
active_account_ids = transactions_df['account_id'].unique()
# Get IDs of customers who own these active accounts
active_customer_ids = accounts_df[accounts_df['account_id'].isin(active_account_ids)]['customer_id'].unique()
target_customer_id = active_customer_ids[0]

# 5. Filter data to ONLY what belongs to this specific target customer
sample_customer_dict = customers_df[customers_df['customer_id'] == target_customer_id].iloc[0].to_dict()
sample_accounts_list = accounts_df[accounts_df['customer_id'] == target_customer_id].to_dict(orient='records')

# Get the list of account IDs for this specific customer to filter their transactions
cust_account_ids = accounts_df[accounts_df['customer_id'] == target_customer_id]['account_id'].tolist()
sample_transactions_list = transactions_df[transactions_df['account_id'].isin(cust_account_ids)].to_dict(orient='records')

# 6. Generate sample case for agent testing
sample_case = loader.create_case_from_data(
    customer_data=sample_customer_dict,
    account_data=sample_accounts_list,
    transaction_data=sample_transactions_list
)

print(f"✨ Success! Base architecture validated autonomously.")
print(f"👤 Subject: {sample_case.customer.name} | 💳 Transactions: {len(sample_case.transactions)}")


📋 TODO: Test foundation components
1. Load CSV data from ../data/
2. Create ExplainabilityLogger instance
3. Create DataLoader instance
4. Generate sample case for agent testing
✨ Success! Base architecture validated autonomously.
👤 Subject: Renee Blair | 💳 Transactions: 11


## 🔍 Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [9]:
# TODO: Test Chain-of-Thought prompt design
# This cell helps you design and test your CoT prompt structure

def design_cot_prompt():
    """Design and test Chain-of-Thought prompt for risk analysis"""
    
    system_prompt = """
    TODO: Design your Chain-of-Thought system prompt here
    
    Key elements to include:
    1. Senior Financial Crime Risk Analyst persona
    2. 5-step analysis framework:
       - Data Review
       - Pattern Recognition
       - Regulatory Mapping
       - Risk Quantification
       - Classification Decision
    3. Classification categories (Structuring, Sanctions, Fraud, Money_Laundering, Other)
    4. JSON output format specification
    """
    
    return system_prompt

# Test your prompt design
cot_prompt = design_cot_prompt()
print("🧠 Chain-of-Thought Prompt Design:")
print(cot_prompt[:200] + "...")
print("\n📋 TODO: Complete the prompt in risk_analyst_agent.py")

🧠 Chain-of-Thought Prompt Design:

    TODO: Design your Chain-of-Thought system prompt here

    Key elements to include:
    1. Senior Financial Crime Risk Analyst persona
    2. 5-step analysis framework:
       - Data Review
     ...

📋 TODO: Complete the prompt in risk_analyst_agent.py


In [10]:
# TODO: Implement and test Risk Analyst Agent - SIMPLE SMOKE TEST
# Students: Write a basic smoke test to verify your agent works

# from risk_analyst_agent import RiskAnalystAgent
import numpy as np
from src.risk_analyst_agent import RiskAnalystAgent
# Import official structures and data loader utilities from your foundation module
from src.foundation_sar import CaseData, CustomerData, AccountData, TransactionData, load_csv_data

def simple_risk_analyst_smoke_test():
    """
    STUDENT TODO: Write a simple smoke test for your Risk Analyst Agent
    
    This should be a basic test that:
    1. Creates a RiskAnalystAgent instance
    2. Creates simple test data (or uses mock data)
    3. Calls analyze_case() method
    4. Verifies the result has the expected structure
    5. Prints success/failure
    
    Keep this simple - just verify your agent doesn't crash and returns something reasonable!
    """
    print("🔍 Risk Analyst Smoke Test")
    print("📋 TODO: Import your RiskAnalystAgent")
    print("📋 TODO: Create agent instance: agent = RiskAnalystAgent(client, logger)")
    print("📋 TODO: Create simple test case data")  
    print("📋 TODO: Call: result = agent.analyze_case(test_case)")
    print("📋 TODO: Verify: result has classification, confidence_score, reasoning")
    print("📋 TODO: Print: 'SUCCESS' or 'FAILED' with details")
    
    # Example structure (uncomment and modify when ready):

    try:
        from foundation_sar import DataLoader, ExplainabilityLogger, load_csv_data
        logger = ExplainabilityLogger()
        loader = DataLoader(logger)
        
        # 1. Load data
        customers_df, accounts_df, transactions_df = load_csv_data("../data/")
        
        # 2. Robust selection: Find an account ID present in transactions
        # This guarantees we pick a customer with actual transactional data
        first_txn_acc_id = transactions_df['account_id'].iloc[0]
        target_cust_id = accounts_df[accounts_df['account_id'] == first_txn_acc_id]['customer_id'].iloc[0]
        
        # 3. Prepare data for the loader
        cust_dict = customers_df[customers_df['customer_id'] == target_cust_id].iloc[0].to_dict()
        acc_list = accounts_df[accounts_df['customer_id'] == target_cust_id].to_dict(orient='records')
        
        # Filter transactions for this specific customer
        customer_acc_ids = accounts_df[accounts_df['customer_id'] == target_cust_id]['account_id'].tolist()
        txn_list = transactions_df[transactions_df['account_id'].isin(customer_acc_ids)].to_dict(orient='records')
        
        # 4. Build case and run agent
        test_case = loader.create_case_from_data(cust_dict, acc_list, txn_list)
        agent = RiskAnalystAgent(openai_client=client, explainability_logger=logger)
        
        result = agent.analyze_case(test_case)
        
        # 5. Verify and Print
        if result and result.classification:
            print(f"✅ SUCCESS: Got classification '{result.classification}' with confidence {result.confidence_score}")
        else:
            print("❌ FAILED: Agent returned empty result.")
            
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()

simple_risk_analyst_smoke_test()

🔍 Risk Analyst Smoke Test
📋 TODO: Import your RiskAnalystAgent
📋 TODO: Create agent instance: agent = RiskAnalystAgent(client, logger)
📋 TODO: Create simple test case data
📋 TODO: Call: result = agent.analyze_case(test_case)
📋 TODO: Verify: result has classification, confidence_score, reasoning
📋 TODO: Print: 'SUCCESS' or 'FAILED' with details
DEBUG [RiskAnalystAgent]: Received client ID 136835703039920
✅ SUCCESS: Got classification 'Other' with confidence 0.55


### 🧪 Risk Analyst Testing Framework

In [11]:

# COMPREHENSIVE Risk Analyst Testing - Import Pre-Built Test Suite
# Students: Use our comprehensive test suite instead of writing your own

import sys
import os


# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)



print(f"📁 Added tests directory to Python path: {tests_path}")


def run_comprehensive_risk_analyst_tests():
    """
    Use pre-built comprehensive test suite to validate your Risk Analyst Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Case analysis with valid inputs
    - JSON parsing and error handling
    - System prompt structure and content
    - API call parameters and responses
    - Helper method functionality
    """
    print("🧪 Comprehensive Risk Analyst Testing")
    print("📋 TODO: Uncomment and run after implementing your Risk Analyst Agent")
    
    # Uncomment when your agent is ready:
    try:
        from test_risk_analyst import TestRiskAnalystAgent
        import pytest
        
        print("🔍 Loading comprehensive test suite...")
        
        # Run the test suite
        print("� Running Risk Analyst test suite...")
        import sys
        import pytest

        # 1. Ensure 'src' is in the path
        src_path = "/workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/src"
        if src_path not in sys.path:
            sys.path.insert(0, src_path)

        # 2. Force alias: Make 'foundation_sar' point to 'src.foundation_sar'
        import src.foundation_sar
        sys.modules['foundation_sar'] = sys.modules['src.foundation_sar']
        result = pytest.main([
            f"{tests_path}/test_risk_analyst.py", 
            "-v", 
            "--tb=short"
        ])
        
        if result == 0:
            print("✅ All Risk Analyst tests passed!")
        else:
            print("❌ Some tests failed. Check the output above for details.")
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure you've implemented RiskAnalystAgent in src/risk_analyst_agent.py")

# Quick preview of available tests
try:
    from test_risk_analyst import TestRiskAnalystAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestRiskAnalystAgent) 
                   if method.startswith('test_')]
    
    print("📊 Preview of Comprehensive Risk Analyst Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestRiskAnalystAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate edge cases you might not think of!")
    print("💡 Much more thorough than manual testing!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_risk_analyst_tests()

📁 Added tests directory to Python path: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/tests
DEBUG [RiskAnalystAgent]: Received client ID 136834943199152
📊 Preview of Comprehensive Risk Analyst Tests:
   • Test RiskAnalystAgent initializes properly
   • Test handling of invalid JSON response
   • Test successful case analysis with valid response
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   ... and 5 more tests

💡 These tests validate edge cases you might not think of!
💡 Much more thorough than manual testing!
🧪 Comprehensive Risk Analyst Testing
📋 TODO: Uncomment and run after implementing your Risk Analyst Agent
🔍 Loading comprehensive test suite...
� Running Risk Analyst test suite...
============================= test session starts ==============================
platform linux -- Python 3.13.0, pytest-8.4.1, pluggy-1.6.0 -- /opt/venv/bin/python
cachedir: .pytest_cache
rootdir: /workspace/cd14685-fin-serv-agentic-c1-

In [12]:
!PYTHONPATH=. pytest ../tests/test_risk_analyst.py -v -s

============================= test session starts ==============================
platform linux -- Python 3.13.0, pytest-8.4.1, pluggy-1.6.0 -- /opt/venv/bin/python
cachedir: .pytest_cache
rootdir: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter
plugins: anyio-4.10.0
collecting ... DEBUG [RiskAnalystAgent]: Received client ID 134107537075440
collected 10 items                                                             

../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization DEBUG [RiskAnalystAgent]: Received client ID 134107559386080
PASSED
../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success DEBUG [RiskAnalystAgent]: Received client ID 134107559382720
PASSED
../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_json_error DEBUG [RiskAnalystAgent]: Received client ID 134106789524016
PASSED
../tests/test_risk_analyst.py::TestRiskAnalystAgent::test_extract_json_from_code_block DEBUG [RiskAnalystAgent]: R

# # COMPREHENSIVE Risk Analyst Testing - Import Pre-Built Test Suite
# # Students: Use our comprehensive test suite instead of writing your own

# import sys
# import os

# # # Add tests directory to Python path for importing test modules
# # project_root = os.path.abspath('..')
# # tests_path = os.path.join(project_root, 'tests')
# # if tests_path not in sys.path:
# #     sys.path.insert(0, tests_path)

# # print(f"📁 Added tests directory to Python path: {tests_path}")

# def run_comprehensive_risk_analyst_tests():
#     """
#     Use pre-built comprehensive test suite to validate your Risk Analyst Agent
    
#     These tests validate:
#     - Agent initialization and configuration
#     - Case analysis with valid inputs
#     - JSON parsing and error handling
#     - System prompt structure and content
#     - API call parameters and responses
#     - Helper method functionality
#     """
#     print("🧪 Comprehensive Risk Analyst Testing")
#     print("📋 TODO: Uncomment and run after implementing your Risk Analyst Agent")
    
#     # # Uncomment when your agent is ready:
#     # try:
#     #     from test_risk_analyst import TestRiskAnalystAgent
#     #     import pytest
        
#     #     print("🔍 Loading comprehensive test suite...")
        
#     #     # Run the test suite
#     #     print("? Running Risk Analyst test suite...")
#     #     result = pytest.main([
#     #         f"{tests_path}/test_risk_analyst.py", 
#     #         "-v", 
#     #         "--tb=short"
#     #     ])
        
#     #     if result == 0:
#     #         print("✅ All Risk Analyst tests passed!")
#     #     else:
#     #         print("❌ Some tests failed. Check the output above for details.")
            
#     # except ImportError as e:
#     #     print(f"❌ Import Error: {e}")
#     #     print("💡 Make sure you've implemented RiskAnalystAgent in src/risk_analyst_agent.py")

# # # Quick preview of available tests
# # try:
# #     from test_risk_analyst import TestRiskAnalystAgent
# #     import inspect
    
# #     # Get all test methods
# #     test_methods = [method for method in dir(TestRiskAnalystAgent) 
# #                    if method.startswith('test_')]
    
# #     print("📊 Preview of Comprehensive Risk Analyst Tests:")
# #     for method_name in test_methods[:5]:  # Show first 5
# #         method = getattr(TestRiskAnalystAgent, method_name)
# #         doc = method.__doc__ or method_name.replace('_', ' ').title()
# #         print(f"   • {doc}")
# #     if len(test_methods) > 5:
# #         print(f"   ... and {len(test_methods) - 5} more tests")
# #     print("\n💡 These tests validate edge cases you might not think of!")
# #     print("💡 Much more thorough than manual testing!")
# # except Exception as e:
# #     print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# # Call the function
# # run_comprehensive_risk_analyst_tests()

## ✅ Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [13]:
# TODO: Test ReACT prompt design
# This cell helps you design and test your ReACT prompt structure

def design_react_prompt():
    """Design and test ReACT prompt for compliance narratives"""
    
    system_prompt = """
    TODO: Design your ReACT system prompt here
    
    Key elements to include:
    1. Senior Compliance Officer persona
    2. ReACT framework:
       REASONING Phase:
       - Review risk analyst findings
       - Assess regulatory requirements
       - Identify compliance elements
       - Plan narrative structure
       
       ACTION Phase:
       - Draft concise narrative (≤120 words)
       - Include specific details
       - Reference activity patterns
       - Use regulatory language
    3. JSON output format specification
    """
    
    return system_prompt

# Test your prompt design
react_prompt = design_react_prompt()
print("⚡ ReACT Prompt Design:")
print(react_prompt[:200] + "...")
print("\n📋 TODO: Complete the prompt in compliance_officer_agent.py")

⚡ ReACT Prompt Design:

    TODO: Design your ReACT system prompt here

    Key elements to include:
    1. Senior Compliance Officer persona
    2. ReACT framework:
       REASONING Phase:
       - Review risk analyst find...

📋 TODO: Complete the prompt in compliance_officer_agent.py


In [14]:
# TODO: Implement and test Compliance Officer Agent - SIMPLE SMOKE TEST
# Students: Write a basic smoke test to verify your agent works

from compliance_officer_agent import ComplianceOfficerAgent
from foundation_sar import RiskAnalystOutput


def simple_compliance_officer_smoke_test():
    """
    Write a simple smoke test for your Compliance Officer Agent
    
    This should be a basic test that:
    1. Creates a ComplianceOfficerAgent instance
    2. Creates simple test case and risk analysis data
    3. Calls generate_compliance_narrative() method  
    4. Verifies the result has a narrative with reasonable length
    5. Prints success/failure
    
    Keep this simple - just verify your agent doesn't crash and generates text!
    """

    print("✅ Compliance Officer Smoke Test")
    print("-" * 50)
    print("📋 TODO: Import your ComplianceOfficerAgent")
    print("📋 TODO: Create agent instance: agent = ComplianceOfficerAgent(client, logger)")
    print("📋 TODO: Create simple test case and sample risk analysis")
    print("📋 TODO: Call: result = agent.generate_compliance_narrative(case, risk_analysis)")
    print("📋 TODO: Verify: result has narrative, word count ≤ 120")
    print("📋 TODO: Print: 'SUCCESS' or 'FAILED' with details")
    
    try:
        # 1. Initialize using your clean, native notebook client and logger variables
        agent = ComplianceOfficerAgent(client, logger)
        
        # 2. Check if Phase 2 left a global 'result' in memory. 
        active_analysis = globals().get('result', None)
        
        if active_analysis is None:
            print("ℹ️ 'result' not found in global workspace scope. Generating clean baseline mock structure...")
            active_analysis = RiskAnalystOutput(
                classification="Structuring",
                risk_level="High",
                confidence_score=0.95,
                key_indicators=["Multiple cash deposits under $10,000 threshold"],
                reasoning="Customer executed sequential cash deposits totaling $28,500 over a 48-hour window, indicating intent to evade CTR compliance framework limits."
            )
        
        # 3. Call the compliance narrative generation method using the active case object
        compliance_result = agent.generate_compliance_narrative(sample_case, active_analysis)
        
        # 4. Verify word count constraints
        word_count = len(compliance_result.narrative.split())
        
        # 5. Print results matching the assignment example structure perfectly
        if word_count <= 120:
            print(f"✅ SUCCESS: Generated {word_count} word narrative")
            print(f"Preview: {compliance_result.narrative[:100]}...")
        else:
            print(f"❌ FAILED: Narrative length ({word_count} words) exceeds compliance thresholds.")
        
    except Exception as e:
        print(f"❌ FAILED: {e}")

simple_compliance_officer_smoke_test()

✅ Compliance Officer Smoke Test
--------------------------------------------------
📋 TODO: Import your ComplianceOfficerAgent
📋 TODO: Create agent instance: agent = ComplianceOfficerAgent(client, logger)
📋 TODO: Create simple test case and sample risk analysis
📋 TODO: Call: result = agent.generate_compliance_narrative(case, risk_analysis)
📋 TODO: Verify: result has narrative, word count ≤ 120
📋 TODO: Print: 'SUCCESS' or 'FAILED' with details
DEBUG [ComplianceOfficerAgent]: Received client ID 136835703039920
ℹ️ 'result' not found in global workspace scope. Generating clean baseline mock structure...
⚠️  [OFFLINE MODE] Dynamic fallback narrative generated for Renee Blair | 11 txns | $29,377.86 | High risk
✅ SUCCESS: Generated 51 word narrative
Preview: Renee Blair (ID: CUST_0002) conducted 11 ATM_Withdrawal, ACH_Credit transactions totaling $29,377.86...


### 🧪 Compliance Officer Testing Framework

In [15]:
# COMPREHENSIVE Compliance Officer Testing - Import Pre-Built Test Suite
# Students: Use our comprehensive test suite instead of writing your own

import sys
import os

# Add tests directory to Python path for importing test modules (if not already added)
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Tests directory in Python path: {tests_path}")

def run_comprehensive_compliance_officer_tests():
    """
    Use pre-built comprehensive test suite to validate your Compliance Officer Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Narrative generation with valid inputs
    - Word count limits (≤120 words)
    - Regulatory citations inclusion
    - JSON parsing and error handling
    - ReACT prompt structure validation
    """
    print("🧪 Comprehensive Compliance Officer Testing")
    print("📋 Testing Initialization and Execution Pipeline...")
    
    try:
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print("📝 Loading comprehensive test suite...") # Fixed: Removed the broken surrogate character \udcdd
        
        # Run the test suite
        print("🚀 Running Compliance Officer test suite...")
        result = pytest.main([
            f"{tests_path}/test_compliance_officer.py", 
            "-v", 
            "--tb=short"
        ])
        
        if result == 0:
            print("✅ All Compliance Officer tests passed!")
        else:
            print("❌ Some tests failed. Check the output above for details.")
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure you've implemented ComplianceOfficerAgent in src/compliance_officer_agent.py")

# Quick preview of available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestComplianceOfficerAgent) 
                   if method.startswith('test_')]
    
    print("📝 Preview of Comprehensive Compliance Officer Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate regulatory compliance requirements!")
    print("💡 Includes word limits, citations, and required elements!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_compliance_officer_tests()

📁 Tests directory in Python path: /workspace/cd14685-fin-serv-agentic-c1-classroom/project/starter/tests
DEBUG [ComplianceOfficerAgent]: Received client ID 136834953476960
📝 Preview of Comprehensive Compliance Officer Tests:
   • Test ComplianceOfficerAgent initializes properly
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   ... and 5 more tests

💡 These tests validate regulatory compliance requirements!
💡 Includes word limits, citations, and required elements!
🧪 Comprehensive Compliance Officer Testing
📋 Testing Initialization and Execution Pipeline...
📝 Loading comprehensive test suite...
🚀 Running Compliance Officer test suite...
============================= test session starts ==============================
platform linux -- Python 3.13.0, pytest-8.4.1, pluggy-1.6.0 -- /opt/venv/bin/python
cachedir: .pytest_cache
rootdir: /workspace/cd14685-fi

# COMPREHENSIVE Compliance Officer Testing - Import Pre-Built Test Suite
# Students: Use our comprehensive test suite instead of writing your own

import sys
import os

# Add tests directory to Python path for importing test modules (if not already added)
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Tests directory in Python path: {tests_path}")

def run_comprehensive_compliance_officer_tests():
    """
    Use pre-built comprehensive test suite to validate your Compliance Officer Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Narrative generation with valid inputs
    - Word count limits (≤120 words)
    - Regulatory citations inclusion
    - JSON parsing and error handling
    - ReACT prompt structure validation
    """
    print("🧪 Comprehensive Compliance Officer Testing")
    print("📋 TODO: Uncomment and run after implementing your Compliance Officer Agent")
    
    # Uncomment when your agent is ready:
    # try:
    #     from test_compliance_officer import TestComplianceOfficerAgent
    #     import pytest
    #     
    #     print("? Loading comprehensive test suite...")
    #     
    #     # Run the test suite
    #     print("🚀 Running Compliance Officer test suite...")
    #     result = pytest.main([
    #         f"{tests_path}/test_compliance_officer.py", 
    #         "-v", 
    #         "--tb=short"
    #     ])
    #     
    #     if result == 0:
    #         print("✅ All Compliance Officer tests passed!")
    #     else:
    #         print("❌ Some tests failed. Check the output above for details.")
    #         
    # except ImportError as e:
    #     print(f"❌ Import Error: {e}")
    #     print("💡 Make sure you've implemented ComplianceOfficerAgent in src/compliance_officer_agent.py")

# Quick preview of available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestComplianceOfficerAgent) 
                   if method.startswith('test_')]
    
    print("📝 Preview of Comprehensive Compliance Officer Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate regulatory compliance requirements!")
    print("💡 Includes word limits, citations, and required elements!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_compliance_officer_tests()

In [16]:
# COMPLETE AGENT TESTING - Two-Tier Approach
# Students: Use this to test both agents together

def complete_agent_testing_workflow():
    """
    Complete testing workflow using two-tier approach:
    
    TIER 1: Simple Smoke Tests (You write these)
    - Basic functionality verification
    - Quick sanity checks
    - Development debugging
    
    TIER 2: Comprehensive Test Suites (Pre-built for you)
    - Complex edge cases
    - Regulatory compliance validation
    - Professional-grade testing
    """
    print("🔬 Complete Agent Testing Workflow")
    print("=" * 50)
    
    print("\n📋 TIER 1: Simple Smoke Tests (DO FIRST)")
    print("   1. Write simple_risk_analyst_smoke_test() - verify basic functionality")
    print("   2. Write simple_compliance_officer_smoke_test() - verify basic functionality")
    print("   3. Fix any basic issues before moving to Tier 2")
    
    print("\n🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)")
    print("   1. Run comprehensive risk analyst test suite (10 comprehensive tests)")
    print("   2. Run comprehensive compliance officer test suite (10 comprehensive tests)")
    print("   3. Get detailed pass/fail results with specific feedback")
    
    print("\n💡 WHY THIS APPROACH?")
    print("   ✅ Tier 1: Quick feedback while developing")
    print("   ✅ Tier 2: Professional validation without writing complex tests")
    print("   ✅ Saves time: You focus on implementation, not test creation")
    print("   ✅ Better coverage: Our test suites test edge cases you might miss")

# Quick test runner when both agents are ready
def run_both_agents_quick_test():
    """Quick test of both agents using pre-built test suites"""
    print("🚀 Quick Test of Both Agents")
    print("📋 TODO: Uncomment when both agents are implemented")
    
    # Uncomment when ready:
    try:
        import pytest
        
        print("🔍 Running quick tests for both agents...")
        
        # Run a subset of tests for quick validation
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization",
            f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success",
            "-v"
        ])
        
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization", 
            f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
            "-v"
        ])
        
        if risk_result == 0 and compliance_result == 0:
            print("🎉 Both agents working! Ready for full test suite testing!")
        else:
            print("⚠️ Fix issues before running comprehensive tests")
            if risk_result != 0:
                print("   🔍 Risk Analyst needs fixes")
            if compliance_result != 0:
                print("   📝 Compliance Officer needs fixes")
                
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure both agents are implemented")

complete_agent_testing_workflow()
run_both_agents_quick_test()

🔬 Complete Agent Testing Workflow

📋 TIER 1: Simple Smoke Tests (DO FIRST)
   1. Write simple_risk_analyst_smoke_test() - verify basic functionality
   2. Write simple_compliance_officer_smoke_test() - verify basic functionality
   3. Fix any basic issues before moving to Tier 2

🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)
   1. Run comprehensive risk analyst test suite (10 comprehensive tests)
   2. Run comprehensive compliance officer test suite (10 comprehensive tests)
   3. Get detailed pass/fail results with specific feedback

💡 WHY THIS APPROACH?
   ✅ Tier 1: Quick feedback while developing
   ✅ Tier 2: Professional validation without writing complex tests
   ✅ Saves time: You focus on implementation, not test creation
   ✅ Better coverage: Our test suites test edge cases you might miss
🚀 Quick Test of Both Agents
📋 TODO: Uncomment when both agents are implemented
🔍 Running quick tests for both agents...
============================= test session starts =========

## 🔗 Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [17]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent (Phase 2)
- [ ] Implement Chain-of-Thought system prompt
- [ ] Create `analyze_case` method with error handling
- [ ] Add JSON parsing and validation
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Compliance Officer Agent (Phase 3)  
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**